In [36]:
import kagglehub
from sklearn.preprocessing import StandardScaler
path = kagglehub.competition_download('playground-series-s4e10')
print(path)


/Users/ignacio/.cache/kagglehub/competitions/playground-series-s4e10


In [42]:
# load the data
import pandas as pd
train_df = pd.read_csv(path + '/train.csv')
test_df = pd.read_csv(path + '/test.csv')    

# print column names
print(train_df.columns)

Index(['id', 'person_age', 'person_income', 'person_home_ownership',
       'person_emp_length', 'loan_intent', 'loan_grade', 'loan_amnt',
       'loan_int_rate', 'loan_percent_income', 'cb_person_default_on_file',
       'cb_person_cred_hist_length', 'loan_status'],
      dtype='str')


In [38]:
# ['id', 'person_age', 'person_income', 'person_home_ownership',
#  'person_emp_length', 'loan_intent', 'loan_grade', 'loan_amnt',
#  'loan_int_rate', 'loan_percent_income', 'cb_person_default_on_file',
#  'cb_person_cred_hist_length', 'loan_status'],

# remove id column
train_df = train_df.drop(columns=['id'])
# change person_home_ownership to categorical with dummy variables and 0,1 for the values
train_df = pd.get_dummies(train_df, columns=['person_home_ownership'], drop_first=True)
# change loan_intent to categorical with dummy variables and 0,1 for the values
train_df = pd.get_dummies(train_df, columns=['loan_intent'], drop_first=True)
# change loan_grade to categorical with dummy variables and 0,1 for the values
train_df = pd.get_dummies(train_df, columns=['loan_grade'], drop_first=True)
# change cb_person_default_on_file to 0,1 for the values
train_df['cb_person_default_on_file'] = train_df['cb_person_default_on_file'].map({'Y': 1, 'N': 0})

# convert boolean dummy columns to int (pandas get_dummies returns bool in newer versions)
bool_cols = train_df.select_dtypes(include='bool').columns
train_df[bool_cols] = train_df[bool_cols].astype(int)

# split into train and test
from sklearn.model_selection import train_test_split
X = train_df.drop(columns=['loan_status'])
y = train_df['loan_status']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# normalize person_age, person_income, person_emp_length, loan_amnt, loan_int_rate, loan_percent_income, cb_person_cred_hist_length to have mean 0 and std 1
scaler = StandardScaler()
X_train[['person_age', 'person_income', 'person_emp_length', 'loan_amnt', 'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length']] = scaler.fit_transform(X_train[['person_age', 'person_income', 'person_emp_length', 'loan_amnt', 'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length']])
X_test[['person_age', 'person_income', 'person_emp_length', 'loan_amnt', 'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length']] = scaler.transform(X_test[['person_age', 'person_income', 'person_emp_length', 'loan_amnt', 'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length']])


# define model architecture
import torch
import torch.nn as nn
import torch.optim as optim

class LoanDefaultNN(nn.Module):
    def __init__(self, input_size, hidden1_size):
        super(LoanDefaultNN, self).__init__()
        # fc means fully connected layer
        self.fc1 = nn.Linear(input_size, 1)

        
    def forward(self, x):
        x = torch.sigmoid(self.fc1(x))
        return x

model = LoanDefaultNN(input_size=X_train.shape[1], hidden1_size=128)
# define loss function and optimizer
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


X_train_tensor = torch.tensor(X_train.values, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)

dataloader = torch.utils.data.DataLoader(
    torch.utils.data.TensorDataset(X_train_tensor, y_train_tensor),
    batch_size=10000,
    shuffle=True,
)

for epoch in range(400):
    for X_batch, y_batch in dataloader:
        optimizer.zero_grad()
        y_hat = model(X_batch)
        loss = criterion(y_hat, y_batch)
        loss.backward()
        optimizer.step()
        print(f"Epoch [{epoch + 1}/10], Loss: {loss.item():.4f}", end="\r")
    print(f"Epoch [{epoch + 1}/10], Loss: {loss.item():.4f}")


# evaluate the model on the test set
model.eval()
with torch.no_grad():
    X_test_tensor = torch.tensor(X_test.values, dtype=torch.float32)
    y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).view(-1, 1)
    y_pred = model(X_test_tensor)
    y_pred_label = (y_pred > 0.5).float()
    accuracy = (y_pred_label == y_test_tensor).float().mean()
    print(f"Test Accuracy: {accuracy.item():.4f}")


Epoch [1/10], Loss: 0.7508
Epoch [2/10], Loss: 0.7458
Epoch [3/10], Loss: 0.7367
Epoch [4/10], Loss: 0.7302
Epoch [5/10], Loss: 0.7247
Epoch [6/10], Loss: 0.7168
Epoch [7/10], Loss: 0.7082
Epoch [8/10], Loss: 0.7000
Epoch [9/10], Loss: 0.6945
Epoch [10/10], Loss: 0.6845
Epoch [11/10], Loss: 0.6783
Epoch [12/10], Loss: 0.6733
Epoch [13/10], Loss: 0.6656
Epoch [14/10], Loss: 0.6589
Epoch [15/10], Loss: 0.6527
Epoch [16/10], Loss: 0.6470
Epoch [17/10], Loss: 0.6419
Epoch [18/10], Loss: 0.6360
Epoch [19/10], Loss: 0.6278
Epoch [20/10], Loss: 0.6246
Epoch [21/10], Loss: 0.6196
Epoch [22/10], Loss: 0.6113
Epoch [23/10], Loss: 0.6049
Epoch [24/10], Loss: 0.6027
Epoch [25/10], Loss: 0.5941
Epoch [26/10], Loss: 0.5937
Epoch [27/10], Loss: 0.5861
Epoch [28/10], Loss: 0.5810
Epoch [29/10], Loss: 0.5793
Epoch [30/10], Loss: 0.5705
Epoch [31/10], Loss: 0.5676
Epoch [32/10], Loss: 0.5610
Epoch [33/10], Loss: 0.5589
Epoch [34/10], Loss: 0.5520
Epoch [35/10], Loss: 0.5508
Epoch [36/10], Loss: 0.5447
E

In [43]:
# generate submission with id and loan_status columns
test_df = pd.get_dummies(test_df, columns=['person_home_ownership'], drop_first=True)
test_df = pd.get_dummies(test_df, columns=['loan_intent'], drop_first=True)
test_df = pd.get_dummies(test_df, columns=['loan_grade'], drop_first=True)
test_df['cb_person_default_on_file'] = test_df['cb_person_default_on_file'].map({'Y': 1, 'N': 0})
bool_cols = test_df.select_dtypes(include='bool').columns
test_df[bool_cols] = test_df[bool_cols].astype(int)
test_df[['person_age', 'person_income', 'person_emp_length', 'loan_amnt', 'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length']] = scaler.transform(test_df[['person_age', 'person_income', 'person_emp_length', 'loan_amnt', 'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length']])
test_tensor = torch.tensor(test_df.drop(columns=['id']).values, dtype=torch.float32)
with torch.no_grad():
    # create new tensor without id column
    test_pred = model(test_tensor)
    test_pred_label = (test_pred > 0.5).float().numpy().flatten()
submission_df = pd.DataFrame({'id': test_df['id'], 'loan_status': test_pred_label})
submission_df.to_csv('submission_loans.csv', index=False)    



In [44]:
!kaggle competitions submit -c playground-series-s4e10 -f submission_loans.csv  -m "Message"

100%|█████████████████████████████████████████| 382k/382k [00:00<00:00, 534kB/s]
Successfully submitted to Loan Approval Prediction